# Chapter 2: Creating and Using a Sequence Dataset

## Introduction

The sequence dataset is a fundamental component of the PG2 dataset system, implemented through the `AssaysDataset` class. It provides functionality for working with biological sequence data and associated measurements (assays). This chapter explains how to create, load, and use sequence datasets in your projects.

## Understanding the AssaysDataset

The `AssaysDataset` class is designed to handle sequence data along with various measurements or assays performed on those sequences. It provides methods for:

1. Loading data from CSV files
2. Accessing sequences and their associated measurements
3. Splitting data into training, validation, and test sets
4. Filtering data by specific targets or engineering rounds

## Creating a Sequence Dataset

There are two main ways to create a sequence dataset:

### 1. From a dataset.toml File

The most common approach is to create a sequence dataset through a `dataset.toml` file:

In [1]:
from pg2_dataset.dataset import Dataset

# First, let's create a simple dataset.toml file
import os

# Define the content of our dataset.toml file
dataset_toml_content = """
[resources]
records = "../example_data/sample_data.csv"  # Path to a CSV file

[records]
columns = ["sequence", "score", "label"]
sequence_feature = "sequence"

[metadata]
name = "Sample Dataset"
description = "A sample dataset for demonstration purposes"
doi = "DOI: 10.1000/example"
source = "Example Source"
xref = ""

[assays.score]
description = "Example score measurement"

[assays.label]
description = "Binary classification label"
"""

# Write the content to a file
with open("sample_dataset.toml", "w") as f:
    f.write(dataset_toml_content)

# Create a sample CSV file
import pandas as pd
import numpy as np

# Create a sample DataFrame
np.random.seed(42)
data = {
    "sequence": [
        "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG",
        "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAAG",
        "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG",
        "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG",
        "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG",
    ],
    "score": np.random.rand(5),
    "label": np.random.choice([0, 1], size=5),
}

df = pd.DataFrame(data)

# Create directory if it doesn't exist
os.makedirs("../example_data", exist_ok=True)

# Save to CSV
csv_path = "../example_data/sample_data.csv"
df.to_csv(csv_path, index=False)

print(f"Created sample CSV file at {os.path.abspath(csv_path)}")
display(df)

# Now load the dataset from the TOML file
try:
    # Load dataset from TOML file
    dataset = Dataset.from_toml("sample_dataset.toml")

    # Access the sequence dataset
    sequence_dataset = dataset.assays

    print(f"\nSuccessfully loaded sequence dataset from TOML file")
    print(f"Number of records: {len(sequence_dataset.records)}")
except Exception as e:
    print(f"Error loading dataset: {e}")

Created sample CSV file at /Users/prashant/Documents/Projects/iff/codebase/pg2-dataset/example_data/sample_data.csv


,sequence,score,label
0,MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYL...,0.374540,0
1,MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYL...,0.950714,0
2,MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYL...,0.731994,0
3,MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYL...,0.598658,0
4,MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYL...,0.156019,1


Error loading dataset: from_toml


### 2. Directly Creating an AssaysDataset

You can also create an `AssaysDataset` directly:

In [2]:
from pg2_dataset.backends import AssaysDataset
from pg2_dataset.primitives.meta import AssaysMeta, SingleAssayMeta

try:
    # Create metadata for assays
    assays_meta = AssaysMeta(
        file_path="../example_data/sample_data.csv",
        sequence_feature="sequence",
        assays={
            "score": SingleAssayMeta(
                description="Example score measurement", features=[]
            ),
            "label": SingleAssayMeta(
                description="Binary classification label", features=[]
            ),
        },
    )

    # Create the dataset
    sequence_dataset = AssaysDataset(meta=assays_meta)

    print(f"Successfully created sequence dataset directly")
    print(f"Number of records: {len(sequence_dataset.records)}")
except Exception as e:
    print(f"Error creating dataset: {e}")

ImportError: cannot import name 'AssaysDataset' from 'pg2_dataset.backends' (/Users/prashant/Documents/Projects/iff/codebase/pg2-dataset/src/pg2_dataset/backends/__init__.py)

## Data Structure

The sequence dataset organizes data into several key components:

### Records

Each record represents a single sequence and its associated measurements:

In [ ]:
try:
    # Access all records in the dataset
    records = sequence_dataset.records

    # Example of a single record
    if records:
        record = records[0]
        print(f"Sequence: {record.sequence}")
        print(f"Engineering round: {record.engineering_round}")
        print(f"Score: {record.model_extra.get('score')}")
        print(f"Label: {record.model_extra.get('label')}")
except Exception as e:
    print(f"Error accessing records: {e}")

### DataFrame Access

You can access the data as a pandas DataFrame:

In [ ]:
try:
    # Get the full DataFrame
    df = sequence_dataset.data_frame
    print("Full DataFrame:")
    display(df.head())

    # Get DataFrame filtered by a specific target
    target_df = sequence_dataset.data_frame_by_target("score")
    print("\nDataFrame filtered by 'score' target:")
    display(target_df.head())
except Exception as e:
    print(f"Error accessing DataFrame: {e}")

## Working with Data Splits

The sequence dataset provides functionality for splitting data into training, validation, and test sets:

### Adding a Split Strategy

In [ ]:
try:
    from pg2_dataset.splits import RandomSplitStrategy

    # Create a split strategy (80% train, 20% validation)
    split_strategy = RandomSplitStrategy(train_ratio=0.6, valid_ratio=0.2)

    # Add the split to the dataset
    sequence_dataset.add_split(
        split_strategy=split_strategy,
        targets=["score"],  # Specific targets to consider
        round_num=1,  # Engineering round to consider
    )

    print("Successfully added split strategy")
except Exception as e:
    print(f"Error adding split strategy: {e}")

### Accessing Split Data

Once splits are defined, you can access the different subsets:

In [ ]:
try:
    # Get training data
    train_data = sequence_dataset.train()
    train_x, train_y = train_data.x, train_data.y

    print(f"Training data shape: X={train_x.shape}, y={train_y.shape}")
    print("Training X data:")
    display(train_x.head())
    print("Training y data:")
    display(train_y.head())

    # Get validation data
    valid_data = sequence_dataset.valid()
    valid_x, valid_y = valid_data.x, valid_data.y
    print(f"\nValidation data shape: X={valid_x.shape}, y={valid_y.shape}")

    # Get test data
    test_data = sequence_dataset.test()
    test_x, test_y = test_data.x, test_data.y
    print(f"Test data shape: X={test_x.shape}, y={test_y.shape}")

    # Get splits for specific targets
    train_data_specific = sequence_dataset.train(targets=["score"])
    print(
        f"\nTraining data for specific target shape: X={train_data_specific.x.shape}, y={train_data_specific.y.shape}"
    )
except Exception as e:
    print(f"Error accessing split data: {e}")

## Working with Engineering Rounds

Many biological datasets involve multiple rounds of engineering or experimentation. The sequence dataset provides methods to work with this structure:

In [ ]:
try:
    # Iterate through datasets by engineering round
    print("Iterating through all engineering rounds:")
    for i, round_df in enumerate(sequence_dataset.iter_by_rounds()):
        print(f"Round {i + 1} data shape: {round_df.shape}")

    # Limit to a maximum round
    print("\nIterating with maximum round limit:")
    for i, round_df in enumerate(sequence_dataset.iter_by_rounds(max_round=1)):
        print(f"Round {i + 1} data shape: {round_df.shape}")
except Exception as e:
    print(f"Error working with engineering rounds: {e}")

## Data Transformation

The sequence dataset handles several data transformations automatically:

### Column Renaming

The dataset automatically renames columns from your CSV to standardized internal names:
- Your sequence column → `"sequence"`
- Your engineering round column → `"engineering_round"`
- Your split column → `"split"`

### Default Values

If certain information is missing:
- Engineering round defaults to 1 if not specified
- Each record gets a unique UUID for tracking

## Example Usage

Here's a complete example of working with a sequence dataset:

In [ ]:
from pg2_dataset.dataset import Dataset
from pg2_dataset.splits import RandomSplitStrategy

try:
    # Load dataset from TOML
    dataset = Dataset.from_toml("sample_dataset.toml")
    sequence_dataset = dataset.assays

    # Examine the data
    print(f"Number of records: {len(sequence_dataset.records)}")
    print(f"Available targets: {sequence_dataset.targets}")
    print(f"Available features: {sequence_dataset.features}")

    # Create a data split
    split_strategy = RandomSplitStrategy(train_ratio=0.7, valid_ratio=0.15)
    sequence_dataset.add_split(split_strategy)

    # Get training and validation data
    train_data = sequence_dataset.train()
    valid_data = sequence_dataset.valid()
    test_data = sequence_dataset.test()

    print(f"\nTraining samples: {len(train_data)}")
    print(f"Validation samples: {len(valid_data)}")
    print(f"Test samples: {len(test_data)}")

    # Access the data
    X_train, y_train = train_data.x, train_data.y
    print(f"\nTraining data shape: X={X_train.shape}, y={y_train.shape}")

    # Example of using the data for a simple model
    print("\nExample of using the data for a simple model:")
    from sklearn.linear_model import LogisticRegression
    import numpy as np

    # Create a simple feature from sequences (just the length)
    X_train_features = np.array([[len(seq)] for seq in X_train["sequence"]])
    X_valid_features = np.array([[len(seq)] for seq in valid_data.x["sequence"]])

    # Train a simple model
    if "label" in y_train.columns:
        model = LogisticRegression()
        model.fit(X_train_features, y_train["label"])

        # Evaluate on validation set
        valid_score = model.score(X_valid_features, valid_data.y["label"])
        print(f"Validation accuracy: {valid_score:.4f}")
    else:
        print("'label' column not found in training data")
except Exception as e:
    print(f"Error in example usage: {e}")

## Best Practices

1. **Data Quality**: Ensure your CSV data has consistent formatting and no missing values in critical columns
2. **Column Naming**: Use clear, consistent naming for sequence and measurement columns
3. **Split Strategies**: Choose appropriate split strategies based on your data characteristics
4. **Engineering Rounds**: If your data involves multiple rounds, ensure the round column is properly specified
5. **Target Selection**: When working with splits, specify only the targets relevant to your analysis

## Troubleshooting

Common issues when working with sequence datasets:

1. **Missing Sequence Data**: Ensure your CSV has a valid sequence column specified in the TOML file
2. **Split Errors**: If you get errors about missing splits, make sure to call `add_split()` before accessing train/valid/test data
3. **Column Mismatch**: Verify that columns specified in your TOML file match those in your CSV
4. **Data Type Errors**: Ensure your measurement data has appropriate types (numeric for measurements, strings for sequences)

## Summary

The sequence dataset provides a powerful and flexible way to work with biological sequence data and associated measurements. By properly configuring your dataset through a TOML file or direct initialization, you can easily load, transform, and split your data for analysis and machine learning tasks.

In [ ]:
# Clean up the files we created
import os

try:
    os.remove("sample_dataset.toml")
    os.remove("../example_data/sample_data.csv")
    print("Cleaned up sample files")
except Exception as e:
    print(f"Error cleaning up: {e}")